# Préparation des données pour la modélisation

## Prérequis

Cette étape repose sur la finalisation de l’analyse exploratoire menée dans le notebook précédent.

Le DataFrame central construit à l’étape 1 a été sauvegardé puis rechargé ici afin de servir de base à la préparation des données pour la modélisation.

## Résultats attendus

L’objectif de ce notebook est d’obtenir :

- `X` : un DataFrame de features prêt à être injecté dans une méthode `fit()` de `scikit-learn` ;
- `y` : une `Series` Pandas correspondant à la variable cible ;
- des fonctions permettant de reproduire les transformations appliquées au DataFrame central.

## Démarche retenue

La préparation des données comprend les opérations suivantes :

1. chargement du DataFrame central ;
2. définition de la cible `y` ;
3. construction d’une première matrice de features `X` ;
4. séparation des variables numériques et catégorielles ;
5. encodage des variables qualitatives ;
6. vérification finale de la compatibilité de `X` et `y` avec `scikit-learn`.

## 1. Imports et chargement des bibliothèques

Cette étape charge les bibliothèques nécessaires à la préparation des données pour la modélisation, notamment pour la gestion des valeurs manquantes et l’encodage des variables catégorielles.## 1. Imports et chargement des bibliothèques

Cette étape charge les bibliothèques nécessaires à la préparation des données pour la modélisation, notamment pour la gestion des valeurs manquantes et l’encodage des variables catégorielles.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder

PROJECT_ROOT = Path.cwd ( ).resolve ( ).parent if Path.cwd ( ).name == "notebooks" else Path.cwd ( ).resolve ( )
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

df_central = pd.read_csv (DATA_PROCESSED / "df_central.csv")

print (df_central.shape)
display (df_central.head ( ))

y = df_central["attrition_bin"].copy ( )

print (y.shape)
print (y.value_counts ( ))

## 2. Point de départ : le DataFrame central

La préparation des données pour la modélisation repose sur le DataFrame central construit à partir des trois sources de données. Ce DataFrame contient à la fois la variable cible et les variables explicatives potentielles.

In [ ]:
df_central["a_quitte_l_entreprise"] = (
    df_central["a_quitte_l_entreprise"]
    .astype (str)
    .str.strip ( )
    .str.capitalize ( )
)

df_central["attrition_bin"] = df_central["a_quitte_l_entreprise"].apply (
    lambda x: 1 if x == "Oui" else 0
)

## 3. Définition de la variable cible

La variable cible retenue pour la modélisation est `attrition_bin` :

- `1` : employé ayant quitté l’entreprise ;
- `0` : employé resté dans l’entreprise.

Cette variable sera utilisée comme cible `y` dans les modèles de classification.

In [ ]:
y = df_central["attrition_bin"].copy ( )

In [ ]:
cols_to_drop = [
    "a_quitte_l_entreprise",
    "attrition_bin",
    "id_employee",
    "code_sondage",
]

X = df_central.drop (columns=[col for col in cols_to_drop if col in df_central.columns]).copy ( )

print (X.shape)
display (X.head ( ))

## 4. Construction d’une première matrice de features

La matrice `X` est construite à partir du DataFrame central en retirant :

- la variable cible textuelle ;
- la variable cible binaire ;
- les identifiants techniques ou administratifs qui n’ont pas de sens prédictif direct.

Cette première version de `X` constitue la base de travail pour les transformations suivantes.

In [ ]:
num_cols = X.select_dtypes (include=["number"]).columns.tolist ( )
cat_cols = X.select_dtypes (include=["object", "string", "category"]).columns.tolist ( )

print ("Colonnes numériques :")
print (num_cols)

print ("\nColonnes catégorielles :")
print (cat_cols)

In [ ]:
def split_features_by_type(df: pd.DataFrame):
    num_cols = df.select_dtypes (include=["number"]).columns.tolist ( )
    cat_cols = df.select_dtypes (include=["object", "string", "category"]).columns.tolist ( )
    return num_cols, cat_cols

## 5. Analyse des valeurs manquantes

Avant la modélisation, il est nécessaire d’identifier les variables contenant des valeurs manquantes.

Les modèles de `scikit-learn` ne peuvent généralement pas traiter directement les données manquantes. Une stratégie d’imputation doit donc être mise en place.

In [ ]:
missing_X = X.isna ( ).sum ( ).sort_values (ascending=False)
display (missing_X[missing_X > 0])

## 5. Analyse des valeurs manquantes

La vérification des valeurs manquantes dans la matrice de features `X` ne met en évidence aucune donnée manquante.

À ce stade, aucune étape d’imputation n’est donc nécessaire avant la modélisation. La préparation pourra se concentrer sur l’encodage des variables catégorielles et la réduction éventuelle des redondances entre features.

In [ ]:
def split_features_by_type(df: pd.DataFrame):
    num_cols = df.select_dtypes (include=["number"]).columns.tolist ( )
    cat_cols = df.select_dtypes (include=["object", "string", "category"]).columns.tolist ( )
    return num_cols, cat_cols

## 6. Identification des variables numériques et catégorielles

Les variables explicatives sont séparées en deux groupes :

- les variables numériques ;
- les variables catégorielles.

Cette distinction est indispensable, car les traitements à appliquer ne sont pas les mêmes selon la nature des variables.

In [ ]:
num_cols, cat_cols = split_features_by_type (X)

print ("Colonnes numériques :", num_cols)
print ("\nColonnes catégorielles :", cat_cols)

## 1. Mise en place de fonctions de transformation

Afin de rendre la préparation des données reproductible et plus robuste, plusieurs fonctions sont définies :

- une fonction pour séparer les variables numériques et catégorielles ;
- une fonction pour construire la cible `y` ;
- une fonction pour construire une première matrice de features `X` ;
- une fonction pour encoder les variables qualitatives.

Cette approche est préférable à une suite d’opérations manuelles répétées, car elle facilite la réutilisation des traitements dans la suite du projet.

In [ ]:
def split_features_by_type(df: pd.DataFrame):
    num_cols = df.select_dtypes (include=["number"]).columns.tolist ( )
    cat_cols = df.select_dtypes (include=["object", "string", "category"]).columns.tolist ( )
    return num_cols, cat_cols


def build_target(df: pd.DataFrame, target_col: str = "attrition_bin") -> pd.Series:
    return df[target_col].copy ( )


def build_feature_matrix(
        df: pd.DataFrame,
        cols_to_drop: list[str] | None = None
) -> pd.DataFrame:
    if cols_to_drop is None:
        cols_to_drop = [
            "a_quitte_l_entreprise",
            "attrition_bin",
            "id_employee",
            "code_sondage",
            "eval_number",
        ]
    return df.drop (columns=[col for col in cols_to_drop if col in df.columns]).copy ( )


def encode_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy ( )
    num_cols, cat_cols = split_features_by_type (df)

    if not cat_cols:
        return df

    encoder = OneHotEncoder (
        drop="first",
        handle_unknown="ignore",
        sparse_output=False
    )

    encoded_array = encoder.fit_transform (df[cat_cols])

    encoded_df = pd.DataFrame (
        encoded_array,
        columns=encoder.get_feature_names_out (cat_cols),
        index=df.index
    )

    return pd.concat ([df[num_cols], encoded_df], axis=1)

## 2. Construction de la cible et de la matrice de features

La variable cible `y` est extraite du DataFrame central à partir de la colonne `attrition_bin`.

La matrice `X` est construite en retirant :

- la cible textuelle ;
- la cible binaire ;
- les identifiants techniques ou administratifs ne présentant pas d’intérêt prédictif direct.

Cette première version de `X` sert de base aux transformations suivantes.

In [ ]:
y = build_target (df_central)
X = build_feature_matrix (df_central)

print ("Shape de y :", y.shape)
print (y.value_counts ( ))

print ("\nShape de X :", X.shape)
display (X.head ( ))

## 3. Identification des types de variables

Les features sont séparées selon leur nature :

- les variables numériques, qui peuvent être utilisées directement dans les analyses de corrélation ;
- les variables catégorielles, qui nécessitent un encodage avant d’être injectées dans un modèle `scikit-learn`.

Cette distinction est essentielle pour choisir une méthode de transformation adaptée à chaque type de donnée.

In [ ]:
num_cols, cat_cols = split_features_by_type (X)

print ("Colonnes numériques :")
print (num_cols)

print ("\nColonnes catégorielles :")
print (cat_cols)

In [ ]:
missing_X = X.isna ( ).sum ( ).sort_values (ascending=False)
display (missing_X[missing_X > 0])

## 4. Vérification des valeurs manquantes

La matrice `X` est contrôlée afin de détecter d’éventuelles valeurs manquantes.

Dans le cas présent, aucune donnée manquante n’est observée. Aucune étape d’imputation n’est donc nécessaire avant la modélisation.

## 5. Encodage des variables qualitatives

Les variables catégorielles ne peuvent pas être injectées directement dans la plupart des modèles de `scikit-learn`.

Un `OneHotEncoder` est donc utilisé afin de transformer les modalités qualitatives en variables indicatrices numériques.

Cette méthode est adaptée ici, car les variables concernées sont nominales et ne portent pas, a priori, d’ordre naturel exploitable directement par le modèle.

In [ ]:
X_prepared = encode_features (X)

print ("Shape de X avant encodage :", X.shape)
print ("Shape de X après encodage :", X_prepared.shape)

display (X_prepared.head ( ))

## 6. Vérification finale de la compatibilité avec `scikit-learn`

Avant de poursuivre vers la modélisation, plusieurs contrôles sont réalisés :

- `X` et `y` doivent avoir le même nombre de lignes ;
- `X` doit être entièrement numérique ;
- aucune valeur manquante ne doit subsister.

Ces vérifications permettent de s’assurer que les données sont bien compatibles avec une méthode `fit()` de `scikit-learn`.

In [ ]:
print ("NaN dans X_prepared :", X_prepared.isna ( ).sum ( ).sum ( ))
print ("NaN dans y :", y.isna ( ).sum ( ))
print ("Shape X_prepared :", X_prepared.shape)
print ("Shape y :", y.shape)

assert len (X_prepared) == len (y), "X et y n'ont pas le même nombre de lignes"
assert X_prepared.isna ( ).sum ( ).sum ( ) == 0, "Il reste des NaN dans X_prepared"
assert y.isna ( ).sum ( ) == 0, "Il reste des NaN dans y"

print ("X_prepared et y sont prêts pour la modélisation.")

## 7. Analyse des corrélations linéaires entre features

Une matrice de corrélation de Pearson est calculée afin d’identifier les variables fortement corrélées entre elles.

Cette étape permet de repérer des redondances potentielles entre features, ce qui peut simplifier la représentation des données et limiter la multicolinéarité dans certains modèles.

In [ ]:
plt.figure (figsize=(14, 10))
corr_matrix = X_prepared.corr (numeric_only=True)
sns.heatmap (corr_matrix, cmap="coolwarm", center=0)
plt.title ("Matrice de corrélation de Pearson des features préparées")
plt.show ( )

In [ ]:
corr_abs = X_prepared.corr (numeric_only=True).abs ( )
upper = corr_abs.where (np.triu (np.ones (corr_abs.shape), k=1).astype (bool))

high_corr_pairs = [
    (col, row, upper.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if pd.notna (upper.loc[row, col]) and upper.loc[row, col] > 0.8
]

high_corr_df = pd.DataFrame (high_corr_pairs, columns=["feature_1", "feature_2", "corr_abs"])
display (high_corr_df.sort_values ("corr_abs", ascending=False))

## 8. Exploration complémentaire des relations non linéaires

En complément de la corrélation de Pearson, un pairplot et une matrice de corrélation de Spearman sont réalisés sur un sous-ensemble de variables quantitatives pertinentes.

Cette approche permet d’explorer des relations qui ne seraient pas uniquement linéaires et d’obtenir une lecture plus fine des dépendances entre certaines variables.

In [ ]:
selected_cols = [
    "age",
    "revenu_mensuel",
    "annee_experience_totale",
    "annees_dans_l_entreprise",
    "distance_domicile_travail",
    "attrition_bin",
]

selected_cols = [col for col in selected_cols if col in df_central.columns]

sns.pairplot (df_central[selected_cols], hue="attrition_bin")
plt.show ( )

In [ ]:
plt.figure (figsize=(8, 6))
spearman_corr = df_central[selected_cols].corr (method="spearman", numeric_only=True)
sns.heatmap (spearman_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title ("Matrice de corrélation de Spearman")
plt.show ( )

In [ ]:
X = X_prepared.copy ( )
y = build_target (df_central)

print ("Shape final de X :", X.shape)
print ("Shape final de y :", y.shape)

display (X.head ( ))
display (y.head ( ))

## Conclusion

Cette étape a permis de transformer le DataFrame central issu de l’analyse exploratoire en un jeu de données exploitable pour la modélisation.

Les résultats attendus sont atteints :

- `X` est un DataFrame de features prêt à être injecté dans une méthode `fit()` de `scikit-learn` ;
- `y` est une `Series` Pandas correspondant à la cible ;
- des fonctions ont été mises en place pour reproduire les principales transformations.

La prochaine étape consistera à entraîner plusieurs modèles de classification et à comparer leurs performances.

## 9. Réduction automatique des features fortement corrélées

Afin de limiter les redondances entre variables explicatives, une suppression automatique des features présentant une corrélation linéaire forte est mise en place.

La logique retenue consiste à :

- calculer la matrice de corrélation absolue ;
- repérer les couples de variables dont la corrélation dépasse un seuil donné ;
- supprimer l’une des deux variables dans chaque couple fortement corrélé.

Cette étape permet d’obtenir une matrice de features plus compacte, tout en conservant l’essentiel de l’information utile à la modélisation.

In [ ]:
def drop_highly_correlated_features(df: pd.DataFrame, threshold: float = 0.8):
    corr_matrix = df.corr (numeric_only=True).abs ( )
    upper = corr_matrix.where (np.triu (np.ones (corr_matrix.shape), k=1).astype (bool))
    to_drop = [column for column in upper.columns if any (upper[column] > threshold)]
    df_reduced = df.drop (columns=to_drop)
    return df_reduced, to_drop

In [ ]:
X_reduced, dropped_corr_features = drop_highly_correlated_features (X_prepared, threshold=0.8)

print ("Nombre de colonnes avant :", X_prepared.shape[1])
print ("Nombre de colonnes après :", X_reduced.shape[1])
print ("Colonnes supprimées pour forte corrélation :")
print (dropped_corr_features)

In [ ]:
display (X_reduced.head ( ))

In [ ]:
X = X_reduced.copy ( )
y = build_target (df_central)

print ("Shape final de X :", X.shape)
print ("Shape final de y :", y.shape)
print ("NaN dans X :", X.isna ( ).sum ( ).sum ( ))
print ("NaN dans y :", y.isna ( ).sum ( ))

### Lecture des résultats

Cette étape permet de réduire automatiquement les redondances les plus fortes entre features.

Le seuil de corrélation absolue a été fixé à `0.8`, ce qui constitue un compromis raisonnable : suffisamment élevé pour ne pas supprimer trop de variables, mais assez strict pour éliminer les dépendances linéaires très marquées.

Le DataFrame `X` obtenu après cette réduction reste entièrement compatible avec une modélisation supervisée sous `scikit-learn`.

### Interprétation de la réduction des corrélations

La suppression automatique n’a retiré qu’un nombre très limité de variables, ce qui indique que la matrice de features préparée ne présente pas de redondance linéaire excessive.

Les variables supprimées correspondent à des features dont l’information semblait déjà largement portée par d’autres variables du jeu de données, notamment après l’encodage des variables catégorielles.

Cette réduction reste modérée et ne remet pas en cause la richesse descriptive de la matrice finale `X`.

In [ ]:
cat_cols

### Analyse du nombre de modalités des variables catégorielles d’origine

Après encodage, les colonnes catégorielles initiales n’existent plus sous leur nom d’origine dans `X`, car elles ont été transformées en variables indicatrices par le `OneHotEncoder`.

Pour analyser le nombre de modalités par variable catégorielle, il faut donc repartir de la matrice de features avant encodage.

In [ ]:
X_original = build_feature_matrix (df_central)
_, cat_cols_original = split_features_by_type (X_original)

modalites_par_colonne = X_original[cat_cols_original].nunique ( ).sort_values (ascending=False)
display (modalites_par_colonne)

In [ ]:
print ("Nombre de colonnes avant encodage :", X_original.shape[1])
print ("Nombre de colonnes après encodage :", X_prepared.shape[1])
print ("Nombre de variables catégorielles d'origine :", len (cat_cols_original))

In [ ]:
print ("Shape final de X :", X.shape)
print ("Shape final de y :", y.shape)
print ("NaN dans X :", X.isna ( ).sum ( ).sum ( ))
print ("NaN dans y :", y.isna ( ).sum ( ))
print (X.dtypes.value_counts ( ))

In [ ]:
modalites_par_colonne = X_original[cat_cols_original].nunique ( ).sort_values (ascending=False)
display (modalites_par_colonne)

In [ ]:
X = build_feature_matrix (df_central)
X_prepared = encode_features (X)
X_reduced, dropped_corr_features = drop_highly_correlated_features (X_prepared, threshold=0.8)

X = X_reduced.copy ( )
y = build_target (df_central)

print ("Shape final de X :", X.shape)
print ("Shape final de y :", y.shape)
print ("NaN dans X :", X.isna ( ).sum ( ).sum ( ))
print ("NaN dans y :", y.isna ( ).sum ( ))
print (X.dtypes.value_counts ( ))

### Vérification finale de la matrice de features

Après exclusion des variables non pertinentes ou trop fortement cardinales, la matrice finale `X` contient **54 variables** pour **1470 observations**.

La cible `y` contient également **1470 lignes**, ce qui confirme l’alignement des données.

Aucune valeur manquante n’est présente dans `X` ni dans `y`, et l’ensemble des features est désormais numérique. Le jeu de données obtenu est donc pleinement compatible avec une modélisation supervisée sous `scikit-learn`.

La réduction de la dimension de `X` améliore également la lisibilité du jeu de données et limite les effets indésirables liés à l’encodage de variables à très forte cardinalité.

## Répartition de la variable cible

Ce graphique permet de vérifier la distribution de la variable cible et d’évaluer l’éventuel déséquilibre entre les employés ayant quitté l’entreprise et ceux qui y sont encore.

Cette vérification est importante avant la phase de modélisation, car une cible déséquilibrée peut influencer le choix des métriques et des stratégies d’entraînement.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

y_plot = y.map ({0: "Non", 1: "Oui"})

plt.figure (figsize=(7, 5))
ax = sns.countplot (x=y_plot)

total = len (y_plot)
for p in ax.patches:
    count = int (p.get_height ( ))
    percent = 100 * count / total
    ax.annotate (
        f"{count}\n({percent:.1f}%)",
        (p.get_x ( ) + p.get_width ( ) / 2, p.get_height ( )),
        ha="center",
        va="bottom"
    )

plt.title ("Répartition de la cible binaire")
plt.xlabel ("Attrition")
plt.ylabel ("Nombre d'employés")
plt.show ( )